## iris数据集

鸢尾花（iris）数据集是一个经典数据集，在统计学习和机器学习领域都经常被用作示例。数据集内包含 3 类共 150 条记录，每类各 50 个数据，每条记录都有 4 项特征：花萼长度、花萼宽度、花瓣长度、花瓣宽度，可以通过这4个特征预测鸢尾花卉属于（iris-setosa, iris-versicolour, iris-virginica）中的哪一品种。

In [ ]:
import seaborn as sns              # 基于 matplotlib 的高级统计可视化库
from pandas import plotting        # pandas 内置的绘图工具（如安德鲁曲线）
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier  # 决策树分类器
from sklearn.datasets import load_iris           # 加载经典 iris 数据集
from sklearn.model_selection import train_test_split  # 数据集划分工具
from sklearn import tree                          # 决策树可视化工具

In [ ]:
# 加载 iris 数据集：150个样本，4个特征，3类鸢尾花
data = load_iris() 
# 转换成 pandas DataFrame 形式，便于数据探索和可视化
df = pd.DataFrame(data.data, columns = data.feature_names)
# 添加品种列（target 为 0/1/2，分别对应三种鸢尾花）
df['Species'] = data.target
# 查看数据集信息（列名、数据类型、非空数量等）
print(f"数据集信息：\n{df.info()}")
# 查看前5条数据，直观了解数据格式
print(f"前5条数据：\n{df.head()}")
# 查看各特征列的摘要统计信息（均值、标准差、分位数等）
df.describe()

通过[Violinplot](https://zhuanlan.zhihu.com/p/376055263)和 Pointplot，分别从数据分布和斜率，观察各特征与品种之间的关系

In [ ]:
# 设置颜色主题，用于区分不同品种的可视化
antV = ['#1890FF', '#2FC25B', '#FACC14', '#223273', '#8543E0', '#13C2C2', '#3436c7', '#F04864'] 

# --- Violinplot（小提琴图）：展示各特征在不同品种下的概率密度分布 ---
# 小提琴图结合了箱线图和核密度估计，能直观看到数据分布形状
f, axes = plt.subplots(2, 2, figsize=(8, 8), sharex=True)
sns.despine(left=True) # 删除上方和右方坐标轴上不需要的边框
sns.violinplot(x='Species', y=df.columns[0], data=df, palette=antV, ax=axes[0, 0])  # 花萼长度
sns.violinplot(x='Species', y=df.columns[1], data=df, palette=antV, ax=axes[0, 1])  # 花萼宽度
sns.violinplot(x='Species', y=df.columns[2], data=df, palette=antV, ax=axes[1, 0])  # 花瓣长度
sns.violinplot(x='Species', y=df.columns[3], data=df, palette=antV, ax=axes[1, 1])  # 花瓣宽度
plt.show()

# --- Pointplot（点图）：展示各特征在不同品种下的均值及置信区间 ---
# 通过斜率可以观察特征与品种之间的线性关系强度
f, axes = plt.subplots(2, 2, figsize=(8, 6), sharex=True)
sns.despine(left=True)
sns.pointplot(x='Species', y=df.columns[0], data=df, color=antV[1], ax=axes[0, 0])
sns.pointplot(x='Species', y=df.columns[1], data=df, color=antV[1], ax=axes[0, 1])
sns.pointplot(x='Species', y=df.columns[2], data=df, color=antV[1], ax=axes[1, 0])
sns.pointplot(x='Species', y=df.columns[3], data=df, color=antV[1], ax=axes[1, 1])
plt.show()

# --- 安德鲁曲线：将高维数据映射为三角函数曲线 ---
# 曲线越相似说明样本越接近，可用于观察不同类别的可分性
plt.subplots(figsize = (8,6))
plotting.andrews_curves(df, 'Species', colormap='cool')
plt.show()

In [ ]:
# --- 决策树模型构建与可视化 ---
# 加载数据集
data = load_iris() 
df = pd.DataFrame(data.data, columns = data.feature_names)
df['Species'] = data.target

# 将数字标签（0/1/2）替换为实际品种名称，便于结果可读
target = np.unique(data.target)
target_names = np.unique(data.target_names)
targets = dict(zip(target, target_names))
df['Species'] = df['Species'].replace(targets)

# 提取特征数据 X 和标签 y
X = df.drop(columns="Species")
y = df["Species"]
feature_names = X.columns  # 记录特征名称，用于可视化
labels = y.unique()        # 记录类别名称

# 划分训练集（60%）和测试集（40%），random_state 保证可复现
X_train, test_x, y_train, test_lab = train_test_split(X,y,
                                                 test_size = 0.4,
                                                 random_state = 42)
# 构建决策树分类器
# max_depth=3：限制树的最大深度为3，防止过拟合
# 决策树通过递归地选择最优划分特征和划分点来构建树结构
# 划分准则默认使用基尼系数(Gini)：Gini(D) = 1 - sum(p_k^2)，衡量数据的不纯度
model = DecisionTreeClassifier(max_depth =3, random_state = 42)
model.fit(X_train, y_train)  # 训练决策树模型
# 以文字形式输出树结构，便于理解决策规则
text_representation = tree.export_text(model)
print(text_representation)
# 用图形方式可视化决策树
# filled=True 表示用颜色填充节点，颜色深浅反映类别纯度
plt.figure(figsize=(30,10), facecolor ='g')
a = tree.plot_tree(model,
                   feature_names = feature_names,
                   class_names = labels,
                   rounded = True,
                   filled = True,
                   fontsize=14)
plt.show()